## Assignment 3: $k$ Nearest Neighbor


**Q1.**
1. What is the difference between regression and classification?
* Regression predicts a numeric outcome
* Classification predicts a category
2. What is a confusion table? What does it help us understand about a model's performance?
* A table that counts predictions by actual class and predicted class. It helps us understand its performance by being able to determine which predictions in the model are correct, and which ones are incorrect.
3. What does the SSE quantify about a particular model?
* It quantifies the errors on a validation set for the observations. This helps us find a k that would best predict a validation set for the minimum validation SSE
4. What are overfitting and underfitting? 
* When the K is too small we have overfitting and it can cause additional noise to be focused on, not the patterns. Underfitting is when the K is too big and can cause the same prediction for every k, not quite capturing the patterns.
5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
* Splitting the data into a training and testing set helps prevent overfitting, because if you evaluate the accuracy on the same set its trained on, it'll cause the model to overfit the data. So this improves the model performance by ensuring that we can accurately prevent overfitting and it does well with unseen data. 
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.
* The strengths of a class label are they let you to quickly and easily make a prediction based on the set outcome/label and its easier for a user to see the decision. A weakness of this is that you don't get to see how close of a prediction was made to put it into said category, like it could've been right on the line and maybe it would be better to be under another label.
* The strength of a probability distribution is that you're able to see how confident the model is about its decision. So based on thresholds you can change it accordingly. A weakness of this however is that its not as easy for people to dechipher the prediction of the model, and it might be harder to understand.

**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
* I selected k by calculating the validation accuracy for each k ranging from 1 to 51. I used the method described in class, except for with classifications, and I got the highest calculated accuracy when k = 2.  
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
* The model is only 45% accurate, I calculated this using the formula given in class. Its highest performance was with mine type 2 with 33/38 of the classifications correct. Then mine type 1, mine type 3, mine type 4, and the least accurate being mine type 5. Mine types 4 and 5 were mostly identified wrongly as mine type 1.
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.
*  I think given the errors I would use this as just something that helps you narrow down your decisions, which could save some time, but I wouldn't trust these decisions completely. This does really good in predicting mine type 2, so I would trust that the most between all of them and wouldn't give as much validation to it. I would defintely have some sort of seperate validation setup for the less accurate mine types.

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** Jalen Medley

**Date:** 10/2/2026

In [31]:
# Your Phase 1 solution to the problem goes here.
import pandas as pd

mines = pd.read_csv("data/land_mines.csv")

print(mines.info())
print(mines.isna().sum())

print(mines['mine_type'].value_counts())

print(mines[['voltage', 'height', 'soil']].describe())

<class 'pandas.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   voltage    338 non-null    float64
 1   height     338 non-null    float64
 2   soil       338 non-null    float64
 3   mine_type  338 non-null    int64  
dtypes: float64(3), int64(1)
memory usage: 10.7 KB
None
voltage      0
height       0
soil         0
mine_type    0
dtype: int64
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64
          voltage      height        soil
count  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550
std      0.195819    0.306043    0.344244
min      0.197734    0.000000    0.000000
25%      0.309737    0.272727    0.200000
50%      0.359516    0.545455    0.600000
75%      0.482628    0.727273    0.800000
max      0.999999    1.000000    1.000000


In [32]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix
import numpy as np

eval_x = mines[['voltage', 'height', 'soil']]
eval_y = mines['mine_type']

eval_x_train, eval_x_valid, eval_y_train, eval_y_valid = train_test_split(eval_x, eval_y, test_size=.5, random_state=65)

eval_scaler = MinMaxScaler()
eval_x_train_scaled = eval_scaler.fit_transform(eval_x_train)
eval_x_valid_scaled = eval_scaler.transform(eval_x_valid)

eval_ks = np.arange(1, 51)
eval_valid_acc = []
eval_train_acc = []

for eval_k in eval_ks:
    eval_candidate = KNeighborsClassifier(n_neighbors=int(eval_k))
    eval_candidate.fit(eval_x_train_scaled, eval_y_train)
    eval_valid_hat = eval_candidate.predict(eval_x_valid_scaled)
    eval_train_hat = eval_candidate.predict(eval_x_train_scaled)
    eval_valid_acc.append(np.mean(eval_y_valid.to_numpy() == eval_valid_hat))
    eval_train_acc.append(np.mean(eval_y_train.to_numpy() == eval_train_hat))

eval_k_start = int(eval_ks[np.argmax(eval_valid_acc)])

print(eval_k_start)
neighbor = KNeighborsClassifier(n_neighbors=eval_k_start).fit(eval_x_train_scaled, eval_y_train)

prediction = neighbor.predict(eval_x_valid_scaled)

confusion = pd.crosstab(eval_y_valid, prediction, rownames=['Actual'], colnames=['Predicted'])

print(confusion)


2
Predicted   1   2   3  4  5
Actual                     
1          21   1   6  4  2
2           0  33   2  3  0
3           5   5  10  6  4
4          13   5  10  9  1
5          10   1   8  7  3


## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** 4fc77528029f6ce049b3e3191c882afb08d7fa87
- **AI tool and model used:** ChatGPT GPT-5.6-Sol

### *Prompts used

1. Given these questions and answers **Insert questions** and the provided code **insert code**, can you provide me a short concise list of suggested changes that ensure accurcy of my solution. If the changes are just nit picky changes that don't change the accuracy don't list them, and if there are no suggestions, just state so.

### *AI-proposed changes


1. Q1 #3 — Clarify what SSE actually measures.
SSE is the sum of the squared differences between actual and predicted numeric values. It does not inherently measure validation error; it can be calculated on training or validation data. You could change it to:
SSE quantifies the total squared error between the actual and predicted values. When calculated on the validation set, we can select the \(k\) with the lowest validation SSE.


2. Q1 #4 — Fix the underfitting explanation slightly.
A large \(k\) does not necessarily cause "the same prediction for every k." Instead, it causes predictions to become overly generalized because too many neighbors influence each prediction:
Underfitting occurs when \(k\) is too large, causing the model to generalize too much and fail to capture important patterns in the data.


3. Q1 #6 — Be careful calling probabilities "confidence."
For k-NN, the probability is based on the proportion of neighboring observations belonging to each class; it is not necessarily a true measure of model confidence. Your basic point is correct, but "probability assigned to each class" would be more accurate than "how confident the model is."

4. Q2 #5 — Don't imply that type 2 predictions can safely receive less validation.
Even though type 2 has the best recall in your table (33/38 actual type-2 mines were correctly identified), that does not mean every prediction of type 2 is highly reliable. In a safety-critical land-mine setting, all predictions should still receive independent verification. You can say type 2 is the model's strongest-performing class, but not that you would validate it less.

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept | I will accept this because it makes the answer to my question clearer, we have discussed this in class and what it quantifies, but maybe my explanation of it isn't 100% accurate to someone who hasn't seen it before. |
| 2 | Accept | I will accept this as well to make my answer between overfitting and underfitting clearer, I feel like I got the general idea of it but fixing this specific will make my answer more precise |
| 3 | Accept | I will accept this as a small fix of wording, my main idea is still correct, I will just be making it more accurate with this word change |
| 4 | Accept | I will accept this because its true, just because it had the highest accuracy of all the mine types, that doesn't mean it shouldn't recieve less validation, I think in me trying to translate my idea I made it seem like we should just drop some of the validation just because its a type 2 mine |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

**Q1.**
1. What is the difference between regression and classification?
* Regression predicts a numeric outcome
* Classification predicts a category
2. What is a confusion table? What does it help us understand about a model's performance?
* A table that counts predictions by actual class and predicted class. It helps us understand its performance by being able to determine which predictions in the model are correct, and which ones are incorrect.
3. What does the SSE quantify about a particular model?
* SSE quantifies the total squared error between the actual and predicted values. When calculated on the validation set, we can select the \(k\) with the lowest validation SSE.
4. What are overfitting and underfitting? 
* When the K is too small we have overfitting and it can cause additional noise to be focused on, not the patterns. Underfitting occurs when \(k\) is too large, causing the model to generalize too much and fail to capture important patterns in the data.

5. Why does splitting the data into training and testing sets, and choosing $k$ by evaluating accuracy or SSE on the test set, improve model performance?
* Splitting the data into a training and testing set helps prevent overfitting, because if you evaluate the accuracy on the same set its trained on, it'll cause the model to overfit the data. So this improves the model performance by ensuring that we can accurately prevent overfitting and it does well with unseen data. 
6. With classification, we can report a class label as a prediction or a probability distribution over class labels. Please explain the strengths and weaknesses of each approach.
* The strengths of a class label are they let you to quickly and easily make a prediction based on the set outcome/label and its easier for a user to see the decision. A weakness of this is that you don't get to see how close of a prediction was made to put it into said category, like it could've been right on the line and maybe it would be better to be under another label.
* The strength of a probability distribution is that you're able to see the probability assigned to each class with its decision. So based on thresholds you can change it accordingly. A weakness of this however is that its not as easy for people to dechipher the prediction of the model, and it might be harder to understand.


**Q2.** This is a case study on $k$ nearest neighbor classification, using the `land_mines.csv` data.

The data consists of a label, `mine_type`, taking integer values 1 to 5, and three properties of the mine, `voltage`, `height` and `soil`. We want to predict the kind of mine from data about it. Imagine working for the DOD or a humanitarian aid agency, trying to help people remove land mines more safely.

1. Load the data. Perform some EDA, summarizing the target label and the features.
2. Split the sample 50/50 into training and test/validation sets. (The smaller the data are, the more equal the split should be, in my experience: Otherwise, all of the members of one class end up in the training or test data, and the model falls apart.)
3. Build a $k$-NN classifier. Explain how you select $k$.
* I selected k by calculating the validation accuracy for each k ranging from 1 to 51. I used the method described in class, except for with classifications, and I got the highest calculated accuracy when k = 2.  
4. Print a confusion table for the optimal model, comparing predicted and actual class label on the test set. How accurate is it? Where is performance more or less accurate?
* The model is only 45% accurate, I calculated this using the formula given in class. Its highest performance was with mine type 2 with 33/38 of the classifications correct. Then mine type 1, mine type 3, mine type 4, and the least accurate being mine type 5. Mine types 4 and 5 were mostly identified wrongly as mine type 1.
5. Notice that you can have a lot of accurate predictions for a given type of mine, but still make a lot of mistakes. Please explain how you'd advise someone to actually use this predictive model in practice, given the errors that it tends to make.
*  I think given the errors I would use this as just something that helps you narrow down your decisions, which could save some time, but I wouldn't trust these decisions completely. This does really good in predicting mine type 2, so this is the model's strongest performing class. I would defintely have some sort of seperate validation setup for the less accurate mine types.

In [33]:
# Your Phase 2 solution to the problem goes here.
# Your Phase 1 solution to the problem goes here.
import pandas as pd

mines = pd.read_csv("data/land_mines.csv")

print(mines.info())
print(mines.isna().sum())

print(mines['mine_type'].value_counts())

print(mines[['voltage', 'height', 'soil']].describe())

<class 'pandas.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   voltage    338 non-null    float64
 1   height     338 non-null    float64
 2   soil       338 non-null    float64
 3   mine_type  338 non-null    int64  
dtypes: float64(3), int64(1)
memory usage: 10.7 KB
None
voltage      0
height       0
soil         0
mine_type    0
dtype: int64
mine_type
1    71
2    70
3    66
4    66
5    65
Name: count, dtype: int64
          voltage      height        soil
count  338.000000  338.000000  338.000000
mean     0.430634    0.508876    0.503550
std      0.195819    0.306043    0.344244
min      0.197734    0.000000    0.000000
25%      0.309737    0.272727    0.200000
50%      0.359516    0.545455    0.600000
75%      0.482628    0.727273    0.800000
max      0.999999    1.000000    1.000000


In [34]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix
import numpy as np

eval_x = mines[['voltage', 'height', 'soil']]
eval_y = mines['mine_type']

eval_x_train, eval_x_valid, eval_y_train, eval_y_valid = train_test_split(eval_x, eval_y, test_size=.5, random_state=65)

eval_scaler = MinMaxScaler()
eval_x_train_scaled = eval_scaler.fit_transform(eval_x_train)
eval_x_valid_scaled = eval_scaler.transform(eval_x_valid)

eval_ks = np.arange(1, 51)
eval_valid_acc = []
eval_train_acc = []

for eval_k in eval_ks:
    eval_candidate = KNeighborsClassifier(n_neighbors=int(eval_k))
    eval_candidate.fit(eval_x_train_scaled, eval_y_train)
    eval_valid_hat = eval_candidate.predict(eval_x_valid_scaled)
    eval_train_hat = eval_candidate.predict(eval_x_train_scaled)
    eval_valid_acc.append(np.mean(eval_y_valid.to_numpy() == eval_valid_hat))
    eval_train_acc.append(np.mean(eval_y_train.to_numpy() == eval_train_hat))

eval_k_start = int(eval_ks[np.argmax(eval_valid_acc)])

print(eval_k_start)
neighbor = KNeighborsClassifier(n_neighbors=eval_k_start).fit(eval_x_train_scaled, eval_y_train)

prediction = neighbor.predict(eval_x_valid_scaled)

confusion = pd.crosstab(eval_y_valid, prediction, rownames=['Actual'], colnames=['Predicted'])

print(confusion)


2
Predicted   1   2   3  4  5
Actual                     
1          21   1   6  4  2
2           0  33   2  3  0
3           5   5  10  6  4
4          13   5  10  9  1
5          10   1   8  7  3


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]

The main improvements of the AI in this assignment, as usual with the previous assignments is just helping me express my ideas more clearly and allowing me to fix wording that caused my answers to not be as clear or as accurate as it should've been despite having the main idea clear. In this aspect I don't really thing there was any mistakes or limiations on its suggestions as it was more of just definitional things and wording fixes, not really much to deal with my code. I verified the final solution of its suggestions by looking back at the slides and the wording that was used on the slides and in class matched what the AI was explaining, especially for SSE. I am most confident in the AI when it provides suggestions like these as opposed to the code suggestions so I don't have any remaining concerns in regards to this.